# Milestone 1: Skin Cancer Dataset Exploration (HAM10000)
**System**: Skin Cancer Detection and Classification System Using CNN-SVM  
**Objective**: Explore dataset labels, class distribution, lesion characteristics, and patient demographics.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from src.preprocessing import CLASS_NAMES, index_image_paths

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})


## 1. Load Metadata and Index Images

In [ ]:
metadata_path = 'dataset/raw/HAM10000_metadata.csv'
if not os.path.exists(metadata_path):
    metadata_path = 'HAM10000_metadata.csv'

df = pd.read_csv(metadata_path)
print(f'Total records: {len(df)}')
print(f'Unique lesions: {df["lesion_id"].nunique()}')
image_map = index_image_paths()
print(f'Total available images indexed: {len(image_map)}')
df.head()

## 2. Analyze Diagnostic Classes & Class Imbalance
The 7 diagnostic classes in HAM10000 are:
- `nv`: Melanocytic Nevi (benign mole)
- `mel`: Melanoma (malignant)
- `bkl`: Benign Keratosis (benign)
- `bcc`: Basal Cell Carcinoma (malignant)
- `akiec`: Actinic Keratoses (pre-malignant / malignant)
- `vasc`: Vascular Lesions (benign)
- `df`: Dermatofibroma (benign)

In [ ]:
class_counts = df['dx'].value_counts()
class_df = pd.DataFrame({
    'Code': class_counts.index,
    'Condition': [CLASS_NAMES[c] for c in class_counts.index],
    'Count': class_counts.values,
    'Percentage': (class_counts.values / len(df) * 100).round(2)
})
display(class_df)

plt.figure(figsize=(10, 4))
sns.barplot(data=class_df, x='Count', y='Condition', palette='viridis')
plt.title('HAM10000 Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Images')
plt.ylabel('Diagnostic Category')
plt.show()

## 3. Patient Demographics & Anatomical Localization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
sns.histplot(df['age'].dropna(), kde=True, ax=axes[0], color='teal', bins=16)
axes[0].set_title('Patient Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age')

# Localization
loc_counts = df['localization'].value_counts()
sns.barplot(x=loc_counts.values, y=loc_counts.index, ax=axes[1], palette='rocket')
axes[1].set_title('Lesion Localization Across Body Sites', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4. Visualize Sample Images from Each Diagnostic Category

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(20, 3))
for idx, (code, name) in enumerate(CLASS_NAMES.items()):
    sample_img_id = df[df['dx'] == code].iloc[0]['image_id']
    img_path = image_map[sample_img_id]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[idx].imshow(img)
    axes[idx].set_title(f"{code.upper()}\n{name}", fontsize=9)
    axes[idx].axis('off')
plt.tight_layout()
plt.show()